[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/#view=upload)

This English notebook is saved locally. The button opens Colab; click **Upload notebook** and select `in-depth-colab_English.ipynb`. It does not automatically open or upload this local file.


# Advanced Study 2. Coding a Neural Network in Python

In [1]:
import random
import numpy as np

# Set environment variables.

# Input and target values
data = [
    [[0, 0], [0]],
    [[0, 1], [1]],
    [[1, 0], [1]],
    [[1, 1], [0]]
]

# Set the number of iterations, learning rate (lr), and momentum coefficient (mo).
iterations=5000
lr=0.1
mo=0.4

# Activation function 1: sigmoid
# Values for the function and its derivative
def sigmoid(x, derivative=False):
    if (derivative == True):
        return x * (1 - x)
    return 1 / (1 + np.exp(-x))

# Activation function 2: tanh
# The derivative of tanh is 1 - (the square of the activation function output).
def tanh(x, derivative=False):
    if (derivative == True):
        return 1 - x ** 2
    return np.tanh(x)

# Function for creating weight arrays
def makeMatrix(i, j, fill=0.0):
    mat = []
    for _ in range(i):
        mat.append([fill] * j)
    return mat

# Running the neural network
class NeuralNetwork:

    # Set initial values.
    def __init__(self, num_x, num_yh, num_yo, bias=1):

        # Input values (num_x), initial hidden layer values (num_yh), initial output layer values (num_yo), and bias
        self.num_x = num_x + bias  # Set the bias to 1 (see the text).
        self.num_yh = num_yh
        self.num_yo = num_yo

        # Initial activation values
        self.activation_input = [1.0] * self.num_x
        self.activation_hidden = [1.0] * self.num_yh
        self.activation_out = [1.0] * self.num_yo

        # Initial input weights
        self.weight_in = makeMatrix(self.num_x, self.num_yh)
        for i in range(self.num_x):
            for j in range(self.num_yh):
                self.weight_in[i][j] = random.random()

        # Initial output weights
        self.weight_out = makeMatrix(self.num_yh, self.num_yo)
        for j in range(self.num_yh):
            for k in range(self.num_yo):
                self.weight_out[j][k] = random.random()

        # Previous weight values for momentum SGD
        self.gradient_in = makeMatrix(self.num_x, self.num_yh)
        self.gradient_out = makeMatrix(self.num_yh, self.num_yo)

    # Update function
    def update(self, inputs):

        # Input layer activations
        for i in range(self.num_x - 1):
            self.activation_input[i] = inputs[i]

        # Hidden layer activations
        for j in range(self.num_yh):
            sum = 0.0
            for i in range(self.num_x):
                sum = sum + self.activation_input[i] * self.weight_in[i][j]
            # Choose sigmoid or tanh as the activation function.
            self.activation_hidden[j] = tanh(sum, False)

        # Output layer activations
        for k in range(self.num_yo):
            sum = 0.0
            for j in range(self.num_yh):
                sum = sum + self.activation_hidden[j] * self.weight_out[j][k]
            # Choose sigmoid or tanh as the activation function.
            self.activation_out[k] = sigmoid(sum, False)

        return self.activation_out[:]
    
    # Run backpropagation.
    def backPropagate(self, targets):

        # Calculate output deltas.
        output_deltas = [0.0] * self.num_yo
        for k in range(self.num_yo):
            error = targets[k] - self.activation_out[k]
            # Choose sigmoid or tanh and apply its derivative.
            output_deltas[k] = sigmoid(self.activation_out[k], True) * error

        # Error function for hidden nodes
        hidden_deltas = [0.0] * self.num_yh
        for j in range(self.num_yh):
            error = 0.0
            for k in range(self.num_yo):
                error = error + output_deltas[k] * self.weight_out[j][k]
                # Choose sigmoid or tanh and apply its derivative.
            hidden_deltas[j] = tanh(self.activation_hidden[j], True) * error

        # Update output weights.
        for j in range(self.num_yh):
            for k in range(self.num_yo):
                gradient = output_deltas[k] * self.activation_hidden[j]
                v = mo * self.gradient_out[j][k] - lr * gradient
                self.weight_out[j][k] += v
                self.gradient_out[j][k] = gradient

        # Update input weights.
        for i in range(self.num_x):
            for j in range(self.num_yh):
                gradient = hidden_deltas[j] * self.activation_input[i]
                v = mo*self.gradient_in[i][j] - lr * gradient
                self.weight_in[i][j] += v
                self.gradient_in[i][j] = gradient

        # Calculate the error (least squares).
        error = 0.0
        for k in range(len(targets)):
            error = error + 0.5 * (targets[k] - self.activation_out[k]) ** 2
        return error

    # Run training.
    def train(self, patterns):
        for i in range(iterations):
            error = 0.0
            for p in patterns:
                inputs = p[0]
                targets = p[1]
                self.update(inputs)
                error = error + self.backPropagate(targets)
            if i % 500 == 0:
                print('error: %-.5f' % error)
    # Print the results.
    def result(self, patterns):
        for p in patterns:
            print('Input: %s, Predict: %s' % (p[0], self.update(p[0])))

if __name__ == '__main__':

    # Configure two input values, two layers, and one output value.
    n = NeuralNetwork(2, 2, 1)

    # Run training.
    n.train(data)

    # Print the results.
    n.result(data)


# Reference: http://arctrix.com/nas/python/bpnn.py (Neil Schemenauer)

error: 0.57628
error: 0.18719
error: 0.06050
error: 0.03095
error: 0.01994
error: 0.01449
error: 0.01129
error: 0.00922
error: 0.00777
error: 0.00670
Input: [0, 0], Predict: [0.04603675806550747]
Input: [0, 1], Predict: [0.9610435797681456]
Input: [1, 0], Predict: [0.9654814269662408]
Input: [1, 1], Predict: [0.08170175146403254]
